The given code is used to conver the Binary files of Demter ICE/IMSC VLF power spectrum data to Ascii Format

The demeter team provided a fortran code in their documentation 

In [1]:
import pandas as pd
import os 
import numpy as np
import struct
from datetime import datetime, date, timedelta 
import tqdm as tqdm
import pickle

In [2]:
RECL = 8510

In [3]:
def getdata(input_file):
    Lt = []
    Al = []
    Lat = []
    Long = []
    dfArray0 = []
    dfArray1 = []
    timel =[]
    # frq =[]
    with open(input_file, 'rb') as f:
        piece = f.read(RECL)
    
        counter = 0  # count of the number of occurrences of RELC
        while piece:
            counter += 1
            datcds = [int.from_bytes(piece[:4], byteorder='big'), int.from_bytes(piece[4:8], byteorder='big')]
            day=np.mod(datcds[0],16777216)
            msec=datcds[1]
            record_date = date(1950, 1, 1) + timedelta(days=int(day))
            i = 8; delta = 2 * 7
            time = struct.unpack('>'+'h'*7, piece[i:i+delta])
            i = i + delta; delta = 2 * 2

            orb, sorb = struct.unpack('>hh', piece[i:i + delta])

            i = i + delta; delta = 8

            tms = "".join([s.decode() for s in struct.unpack('>' +'s'*8, piece[i:i+delta])])

            i = i + delta; delta = 4
            soft_cal = struct.unpack('>bbbb', piece[i:i + delta])

            i = i + delta; delta = 4*4
            orbp = struct.unpack('>ffff', piece[i:i+delta])
            Geocentric_lat = f"{orbp[0]:.4f}"
            Geocentric_long = f"{orbp[1]:.4f}"
            local_time = f"{orbp[3]:.4f}"
            Altitude = f"{orbp[2]:.3f}"
            i += delta; delta = 4*15
            geop = struct.unpack('>'+'f'*15, piece[i:i+delta])


            i += delta; delta = 4*3
            solp = struct.unpack('>'+'f'*3, piece[i:i+delta])

            i = i + delta; delta = 2
            vers = struct.unpack('>bb', piece[i:i + delta])

            i += delta; delta = 4*18
            attp = struct.unpack('>'+'f'*18, piece[i:i+delta])

            i = i + delta; delta = 2
            qi = struct.unpack('>h', piece[i:i + delta])[0]

            i = i + delta; delta = 2
            vers = struct.unpack('>bb', piece[i:i + delta])

            i = i + delta; delta = 21
            type = "".join([s.decode() for s in struct.unpack('>' +'s'*21, piece[i:i+delta])])

            i = i + delta; delta = 32
            hk = struct.unpack('>'+'c'*32, piece[i:i + delta])
            HKs = "".join([el.hex() for el in hk])

            i = i + delta; delta = 9
            coord = "".join([s.decode() for s in struct.unpack('>' +'s'*9, piece[i:i+delta])])

            i = i + delta; delta = 3
            name = "".join([s.decode() for s in struct.unpack('>' +'s'*3, piece[i:i+delta])])

            i = i + delta; delta = 16
            unit = "".join([s.decode('latin-1') for s in struct.unpack('>' +'s'*16, piece[i:i+delta])])

            i = i + delta; delta = 1
            nbsp = struct.unpack('>'+'b'*1, piece[i:i + delta])[0]
            #print(f"Number of consecutice spectra = {nbsp}")
            i = i + delta; delta = 2
            nbf = struct.unpack('>h', piece[i:i + delta])[0]
            #print(f"Number of spectrum frequencies  = {nbf}")
            i += delta; delta = 4
            dt = struct.unpack('>f', piece[i:i+delta])[0]
            
            #print(f"Time duration of one data array = {dt:.4f}")
            i += delta; delta = 4
            freq = struct.unpack('>f', piece[i:i+delta])[0]
            Frequency_resolution = f"{freq:.4f}"
            #print(f"Frequency resolution = {freq:.4f}")
            i += delta; delta = 4*2
            frange = struct.unpack('>ff', piece[i:i+delta])[0]
            #print(f"Frequency range = {frange:.4f}")
            i += delta; delta = 2 * 7
            idatsp = struct.unpack('>'+'h'*7, piece[i:i+delta])
            # print(f"time = {idatsp}")
            i += delta
            sp = struct.unpack('>2048f', piece[i:])
            sp_0 = sp[:1024]
            sp_1 = sp[1024:]
            dfArray0.append(np.array(sp_0)) 
            dfArray1.append(np.array(sp_1)) 
            dt = datetime(year=int(time[0]), month=int(time[1]), day=int(time[2]), hour=int(time[3]),minute=int(time[4]), second=int(time[5]),microsecond=int(time[6])*1000)
            date_time = dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
            timel.append(date_time)
            Lt.append(local_time)
            Al.append(Altitude)
            Lat.append(Geocentric_lat)
            Long.append(Geocentric_long)
            #frq.append(Frequency_resolution)
           
            piece = f.read(RECL)
            # print(dfArray0)
    return  timel, Lat, Long,Lt, Al, dfArray0, dfArray1,counter
                        

In [4]:
def consolidateDataframes(timel , Lat , Long, Lt, Al, sp0, sp1 ):
    OP = pd.DataFrame({'time':timel,'Geoc_lat':Lat,'Geoc_long':Long,'Local_time': Lt, 'Altitude':Al, 'spectrum0': sp0,'spectrum1': sp1}) 
    df2 = pd.concat([OP])
    
    df2.columns=['datetime', 'geoc_lat', 'geoc_long','local_time', 'altitude','spectrum0','spectrum1' ]
    #fre = df2.Frequency_Resolution.values
    return df2


For a single Folder 

In [ ]:
DFlist =[]
count = []
# loop through all files in the folder
folder_path = "E:\IMSC-DAT"
RECL = 8510
# loop through all files in the folder
for filename in tqdm(os.listdir(folder_path)):
    Data =list(getdata(os.path.join(folder_path,filename)))
    P = consolidateDataframes(Data[0],Data[1],Data[2],Data[3],Data[4],Data[5],Data[6])
    output_filename = os.path.splitext(filename[7:])[0] + "_full"+".pkl"
    with open(os.path.join("E:\\ASCII-IMSC", output_filename), "wb") as k:
        pickle.dump(P, k) #each files will be saved in its corresponding name
    DFlist.append(P) # to save all file in to single file( for example in case of a spectrum of a day (16 orbits a day-concat them together))
    count.append(Data[7])
    #print(f'the count of repeatation of relc : {Data[7]}')
print(sum(count))
result_df = pd.concat(DFlist, axis=0, ignore_index=True)
# result_df.to_pickle('D:\PhD-FBK\ASCII\Orits-together.pkl')

I had made a cleaning based on the missing orbits and Finally i made a list of all up and down orbits (which completes a full orbit) and saved it in the file I_IMSC/ICE_CleanINPUT_Orbits.csv 

In [5]:

path_clean = "E:\Files-firstYear\IMSC-LIST\I_IMSC_CleanINPUT_Orbits.csv"
# since i need the ASCII conversion of files in the cleaned orbits, i am uing this methode.
def halforbits(orbitfile):
    up_fn = orbitfile['UpOrbits']
    dn_fn = orbitfile['DownOrbits']
    half_orbits = []

    for uporb in up_fn:
        half_orbits.append(uporb)

    for dnorb in dn_fn:
        half_orbits.append(dnorb)

    # Create a DataFrame with the "Half Orbits" column
    df = pd.DataFrame({"Half Orbits": half_orbits})

    return df

orbitfile = pd.read_csv(path_clean)
data = halforbits(orbitfile)
data = data[:-1]
data

<>:1: SyntaxWarning: invalid escape sequence '\F'
<>:1: SyntaxWarning: invalid escape sequence '\F'
C:\Users\mbabu\AppData\Local\Temp\ipykernel_23440\598251895.py:1: SyntaxWarning: invalid escape sequence '\F'
  path_clean = "E:\Files-firstYear\IMSC-LIST\I_IMSC_CleanINPUT_Orbits.csv"


I have saved datas in its corresponing year foler insider a parent folder named ICE-DAT/IMSC-DAT

In [7]:
parent_folder ="E:\IMSC-DAT"
count = []
for foldername in os.listdir(parent_folder):
    current_folder = os.path.join(parent_folder, foldername)
    
    # if os.path.isdir(current_folder):
    for filename in tqdm.tqdm(os.listdir(current_folder)):
        Data =list(getdata(os.path.join(current_folder,filename)))
            # print(Data[0].shape)
        P = consolidateDataframes(Data[0],Data[1],Data[2],Data[3],Data[4],Data[5],Data[6])
        count.append(Data[7])
        output_filename = os.path.splitext(filename[7:])[0] + "_full"+".pkl"
        with open(os.path.join("E:\\ASCII-IMSC", output_filename), "wb") as k:
            pickle.dump(P, k)

    print(len(count))

<>:1: SyntaxWarning: invalid escape sequence '\I'
<>:1: SyntaxWarning: invalid escape sequence '\I'
C:\Users\mbabu\AppData\Local\Temp\ipykernel_23440\1399773042.py:1: SyntaxWarning: invalid escape sequence '\I'
  parent_folder ="E:\IMSC-DAT"
  0%|          | 0/8230 [00:00<?, ?it/s]

 54%|█████▍    | 4452/8230 [6:55:04<5:52:13,  5.59s/it] 
C:\Users\mbabu\AppData\Local\Temp\ipykernel_23440\1399773042.py:1: SyntaxWarning: invalid escape sequence '\I'
  parent_folder ="E:\IMSC-DAT"


KeyboardInterrupt: 

check how renaming works 

In [ ]:

original_string = "1132_026560_20050101_014337_20050101_021752.pkl"

# Split the string based on underscore
parts = original_string.split('.')

# Create the new formatted string
new_string = '_'.join(['DMT_N1'] + parts[0:1]) + ".DAT"

# Print the result
print(new_string)
parts

DMT_N1_1132_026560_20050101_014337_20050101_021752.DAT


['1132_026560_20050101_014337_20050101_021752', 'pkl']